# Lab 01: Your First LangGraph Workflow

**Goal:** Build a simple graph with state, nodes, and edges to understand the core building blocks of LangGraph.

**What you'll learn:**
- How to define state with TypedDict
- How to create nodes (Python functions that update state)
- How to connect nodes with edges (START → node → END)
- How to compile and invoke a graph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

## Step 1: Define the State

State is a TypedDict — a dictionary with typed fields.
Every node in the graph reads from and writes to this state.

In [ ]:
class GreetingState(TypedDict):
    name: str
    greeting: str

print("GreetingState has two fields: 'name' (str) and 'greeting' (str)")

## Step 2: Create a Node

A node is a Python function that:
- Takes the full state as input
- Returns a dict with the fields to update

In [ ]:
def greet(state: GreetingState) -> dict:
    """Generate a greeting for the given name."""
    name = state["name"]
    return {"greeting": f"Hello, {name}! Welcome to UniGPS."}

print("Function 'greet' takes state and returns a greeting update")

## Step 3: Build the Graph

1. Create a StateGraph with the state type
2. Add nodes
3. Connect them with edges

In [ ]:
graph = StateGraph(GreetingState)
graph.add_node("greet", greet)
graph.add_edge(START, "greet")  # START → greet
graph.add_edge("greet", END)    # greet → END

print("Graph: START → greet → END")

## Step 4: Compile and Run

compile() creates a runnable application from the graph.
invoke() runs the graph with initial state values.

In [ ]:
app = graph.compile()

result = app.invoke({"name": "Priya"})

print(f"Input:  name = 'Priya'")
print(f"Output: {result}")
print(f"Greeting: {result['greeting']}")

## Step 5: Try with Different Inputs

In [ ]:
for name in ["Rahul", "Anita", "Vikram"]:
    result = app.invoke({"name": name})
    print(f"  {name} → {result['greeting']}")

## Step 6: Two-Node Graph

Let's add a second node that modifies the greeting.

In [ ]:
class FormalState(TypedDict):
    name: str
    department: str
    greeting: str

def create_greeting(state: FormalState) -> dict:
    return {"greeting": f"Welcome, {state['name']}!"}

def add_department(state: FormalState) -> dict:
    return {"greeting": f"{state['greeting']} You are in the {state['department']} department."}

graph2 = StateGraph(FormalState)
graph2.add_node("greet", create_greeting)
graph2.add_node("add_dept", add_department)
graph2.add_edge(START, "greet")
graph2.add_edge("greet", "add_dept")   # greet → add_dept
graph2.add_edge("add_dept", END)

app2 = graph2.compile()

print("Graph: START → greet → add_dept → END")
result = app2.invoke({"name": "Priya", "department": "Engineering"})
print(f"Result: {result['greeting']}")

result = app2.invoke({"name": "Rahul", "department": "Sales"})
print(f"Result: {result['greeting']}")

## TODO 1: Add a Third Node

Add a node called "add_office" that appends the office location
to the greeting. Use a new state field "office" (str).
The graph should be: START → greet → add_dept → add_office → END
Test with: `{"name": "Anita", "department": "QA", "office": "Pune"}`

In [ ]:
# class ExtendedState(TypedDict):
#     name: str
#     department: str
#     office: str
#     greeting: str
#
# def create_greeting_v2(state): return {"greeting": f"Welcome, {state['name']}!"}
# def add_department_v2(state): return {"greeting": f"{state['greeting']} Dept: {state['department']}."}
# def add_office(state): return {"greeting": f"{state['greeting']} Office: {state['office']}."}
#
# graph3 = StateGraph(ExtendedState)
# ... add nodes and edges ...
# app3 = graph3.compile()
# result = app3.invoke({"name": "Anita", "department": "QA", "office": "Pune"})
# print(f"Extended: {result['greeting']}")

## TODO 2: Experiment with Node Order

What happens if you swap the edge order so add_dept runs
BEFORE greet? Does the greeting still make sense?
Try: START → add_dept → greet → END
What does the output look like?

In [ ]:
# Try swapping the edge order and observe what happens

## Key Takeaways

- State = TypedDict defining the data schema
- Nodes = Python functions that read state and return updates
- Edges = connections: START → nodes → END
- compile() builds the graph, invoke() runs it
- Nodes only update the fields they return